# Diabetes Data Analysis & Visualization
**Objective:** Explore the supplied diabetes dataset, assess data quality, prepare analysis-ready data, and communicate patterns with charts and SQL.

> Educational analysis only; not a medical diagnostic tool.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/diabetes_raw.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())

## 1. Data quality checks

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe().T)

## 2. Clean the data
In this dataset, zero values in Glucose, BloodPressure, SkinThickness, Insulin, and BMI are treated as missing measurements for this analysis. Median imputation is used to retain records.

In [ ]:
clean = df.copy()
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_as_missing:
    clean[col] = clean[col].replace(0, np.nan)
clean[zero_as_missing] = clean[zero_as_missing].fillna(clean[zero_as_missing].median())
clean.to_csv("data/diabetes_cleaned.csv", index=False)
display(clean.isna().sum().to_frame("missing_after_cleaning"))
display(clean.head())

## 3. Exploratory data analysis

In [ ]:
display(df["Outcome"].value_counts().rename(index={0:"Outcome 0", 1:"Outcome 1"}))
print("Outcome 1 percentage:", round((df["Outcome"] == 1).mean()*100, 2), "%")

In [ ]:
df["Outcome"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0,1], ["Outcome 0", "Outcome 1"], rotation=0)
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome"); plt.ylabel("Records"); plt.tight_layout(); plt.show()

In [ ]:
df.groupby("Outcome")[["Glucose", "BMI", "Age"]].mean().round(2)

In [ ]:
df.groupby("Outcome")["Glucose"].mean().plot(kind="bar")
plt.xticks([0,1], ["Outcome 0", "Outcome 1"], rotation=0)
plt.title("Average Glucose by Outcome")
plt.xlabel("Outcome"); plt.ylabel("Average Glucose"); plt.tight_layout(); plt.show()

In [ ]:
df.boxplot(column="BMI", by="Outcome")
plt.title("BMI Distribution by Outcome"); plt.suptitle("")
plt.xlabel("Outcome (0 = No, 1 = Yes)"); plt.ylabel("BMI")
plt.tight_layout(); plt.show()

In [ ]:
corr = clean.corr(numeric_only=True)
plt.figure(figsize=(10,7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

## 4. Business-style questions
1. How are records distributed across the two outcome labels?
2. How do average glucose and BMI differ between outcome groups?
3. Which numeric variables show stronger linear relationships with the target?
4. How does the number of outcome-1 records vary by age group?

These are descriptive questions. Associations in this dataset do not establish causation.

In [ ]:
clean["AgeGroup"] = pd.cut(clean["Age"], bins=[0,29,39,49,120],
                          labels=["Under 30","30-39","40-49","50+"])
age_summary = clean.groupby("AgeGroup", observed=False).agg(
    records=("Outcome","size"),
    outcome_1_records=("Outcome","sum")
).reset_index()
display(age_summary)

## 5. Conclusion
Summarize the patterns observed in the tables and charts. Avoid presenting the analysis as a diagnosis or claiming that any feature causes diabetes.